# Documents and Tables in One Graph

A PDF knows that some company missed its revenue guidance. A CSV knows how old that
company's employees are. Neither can answer **"who works at the company that reported a
shortfall, and how old are they?"** — the graph can, once both halves land on the same nodes.

This notebook loads a document, a table, another document and another table, a transcript
that spells a name differently, then a corrected version of the first table. Stop at any checkpoint and look at the graph in the
FalkorDB browser at http://localhost:3000.

```
docker run -p 6379:6379 -p 3000:3000 falkordb/falkordb
export OPENAI_API_KEY="sk-..."
```

In [1]:
import os
import shutil
import sys
import tempfile
from pathlib import Path

# Use THIS checkout, not whatever `pip install graphrag-sdk` left in the environment.
# Two things go wrong otherwise, and both have been seen:
#   1. an older installed copy has no `Column`/`TableMapping`, so the import fails;
#   2. if that failed import already ran once in this kernel, the old module is
#      cached in sys.modules and a sys.path change alone does nothing.
# So: find the checkout's src/ by walking up from here, put it first, and evict any
# graphrag_sdk already imported from somewhere else.
_here = Path.cwd().resolve()
_src = next(
    (d / "src" for d in (_here, *_here.parents) if (d / "src" / "graphrag_sdk").is_dir()),
    None,
)
if _src is not None:
    sys.path.insert(0, str(_src))
    for _m in [m for m in sys.modules if m == "graphrag_sdk" or m.startswith("graphrag_sdk.")]:
        del sys.modules[_m]

from graphrag_sdk import (  # noqa: E402
    Column,
    ConnectionConfig,
    Entity,
    GraphRAG,
    Link,
    LiteLLM,
    LiteLLMEmbedder,
    Ontology,
    TableMapping,
)

_loaded = Path(sys.modules["graphrag_sdk"].__file__).resolve().parent
print(f"graphrag_sdk from {_loaded}")
if _src is not None and _src.resolve() not in _loaded.parents:
    raise RuntimeError(
        f"Loaded graphrag_sdk from {_loaded}, not from this checkout. "
        "Restart the kernel (Kernel -> Restart) and run this cell first."
    )

# Load the repo .env so OPENAI_API_KEY is set without an export. An already-exported
# key wins (load_dotenv never overrides); the connection below is explicit, so the
# FALKOR_* entries in that file are not used here.
from dotenv import load_dotenv  # noqa: E402

for _envfile in (Path.home() / "workspaces/GraphRAG-SDK/.env", Path(".env"), Path("../../.env")):
    if _envfile.is_file():
        load_dotenv(_envfile)
        break
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY not found: put it in the repo .env"

FIXTURES = Path("data/hybrid_walkthrough")
USED = ["board_review.pdf", "employees.csv", "market_note.pdf",
        "organizations.csv", "contracts.csv", "employees_v2.csv"]

# Copied, because the last step rewrites employees.csv.
WORK = Path(shutil.copytree(FIXTURES, Path(tempfile.mkdtemp(prefix="graphrag-")) / "data"))
for name in USED:
    print(f"  {name:22} {(WORK / name).stat().st_size:>6} bytes")

graphrag_sdk from /home/shubeli/workspaces/copilot-worktrees/GraphRAG-SDK/structured-ingestion/graphrag_sdk/src/graphrag_sdk
  board_review.pdf         1962 bytes
  employees.csv             275 bytes
  market_note.pdf          1422 bytes
  organizations.csv         174 bytes
  contracts.csv             203 bytes
  employees_v2.csv          284 bytes


## 1. Say what the tables mean

A mapping turns one row into one entity, and it lives in the **ontology** next to the entity
types. That placement matters: the labels and column types it declares are registered
*before* any prose is extracted, so the extractor uses the real labels instead of guessing —
which is what makes the load order below irrelevant.

`source` is the filename. It is what `ingest` matches on, and it is also where each
property's **signature** comes from: `age` declared by `employees.csv` is stored as
`employees__age`.

Note `Column("age", "INTEGER")` rather than the bare `"age"` shorthand. The shorthand means
STRING, and a STRING column cannot be averaged — the SDK warns at ingest if a column you
declared STRING turns out to hold only numbers.

In [2]:
ONTOLOGY = Ontology(
    entities=[Entity(label="Person"), Entity(label="Organization")],
    tables=[
        TableMapping(
            source="employees.csv",
            label="Person",
            key="employee_id",       # for links and re-sync; the node id comes from name
            name="full_name",        # the identity: same id a prose mention gets
            properties={
                "age": Column("age", "INTEGER"),
                "title": Column("job_title"),
            },
            links=[Link("WORKS_AT", to="Organization", by="org_id")],
        ),
        TableMapping(
            source="organizations.csv",
            label="Organization",
            key="org_id",
            name="org_name",
            properties={"country": Column("country")},
        ),
        # A new label, and two links out of ONE row. `Contract` is not in `entities`
        # above, so the mapping has to say how it connects or construction is refused.
        TableMapping(
            source="contracts.csv",
            label="Contract",
            key="contract_id",
            name="contract_name",
            properties={"value_musd": Column("value_musd", "FLOAT")},
            links=[
                Link("BUYER", to="Organization", by="buyer_org_id"),
                Link("SELLER", to="Organization", by="seller_org_id"),
            ],
        ),
    ],
)

for t in ONTOLOGY.tables:
    print(f"  {t.source:20} -> :{t.label:13} signs {t.signature!r}")

  employees.csv        -> :Person        signs 'employees'
  organizations.csv    -> :Organization  signs 'organizations'
  contracts.csv        -> :Contract      signs 'contracts'


## 2. Connect

`enable_cypher=True` is what makes *"what is the average age"* answerable: it turns the
question into a query over the declared column instead of hunting for a passage that states
the average.

`finalize()` does more than merge spellings. By default it builds an `LLMVerifiedResolution`
over `LLM` and `EMBEDDER` — embed the names, ask the model about the close pairs — and lets it
judge the **whole graph**, which is the only place a table's "Priya Raman" and a note's
"Ms. Raman" ever meet. It decides identity; the merge keeps the table's node and every value
the table signed onto it. `finalize(resolve=False)` would report those pairs instead of asking.


In [3]:
LLM = LiteLLM(model="openai/gpt-4o-mini", api_key=os.environ["OPENAI_API_KEY"], temperature=0.0)
EMBEDDER = LiteLLMEmbedder(
    model="openai/text-embedding-3-small",
    api_key=os.environ["OPENAI_API_KEY"],
    dimensions=256,
)
rag = GraphRAG(
    connection=ConnectionConfig(host="localhost", graph_name="documents_and_tables"),
    llm=LLM,
    embedder=EMBEDDER,
    embedding_dimension=256,
    ontology=ONTOLOGY,
    enable_cypher=True,
)

await rag.delete_all()   # so the notebook re-runs from the top


async def show(label):
    n = (await rag.query("MATCH (e:__Entity__) RETURN count(e)"))[0][0]
    r = (await rag.query("MATCH ()-[x:RELATES]->() RETURN count(x)"))[0][0]
    print(f"{label}: {n} entities, {r} relationships")


async def report(label):
    """Everything finalize() found. Nothing here is decided by a threshold: a pair
    merges because it spelled one name two ways, or because the model said so."""
    summary = await rag.finalize()
    print(f"{label}: merged {summary.entities_deduplicated}, "
          f"embedded {summary.entities_embedded} entities")
    for field in ("resolved_duplicates", "rejected_duplicates", "property_conflicts",
                  "probable_duplicates", "proposed_mappings",
                  "unresolved_references", "stale_signed_properties", "mapping_changed",
                  "entities_without_a_name", "unmerged_name_collisions"):
        value = getattr(summary, field)
        if value:
            print(f"    {field}: {value}")
    return summary


async def ask(question):
    answer = await rag.completion(question)
    print(f"Q: {question}\nA: {answer.answer}")

## 3. A document, then a table

`employees.csv` links every person to an organization by `org_id`, and no organization has
been loaded yet. Those become **reference nodes**: real and keyed, but named by their key
until the source that owns them arrives.

In [4]:
await rag.ingest(str(WORK / "board_review.pdf"))
await rag.ingest(str(WORK / "employees.csv"))
await report("finalize")
await show("after the first document and table")

/home/shubeli/miniconda3/envs/games/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/shubeli/miniconda3/envs/games/lib/python3.10/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/home/shubeli/miniconda3/envs/games/lib/python3.10/site-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(



Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]


Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 1256.08it/s]

Non-transient FalkorDB query failure: ResponseError: Attribute 'id' is already indexed


...

Non-transient FalkorDB query failure: ResponseError: Attribute 'embedding' is already indexed


Non-transient FalkorDB query failure: ResponseError: Attribute 'text' is already indexed


Non-transient FalkorDB query failure: ResponseError: Attribute 'name' is already indexed


finalize: reference targets no source ever described: Organization x2


finalize: merged 0, embedded 9 entities
    unresolved_references: {'Organization': 2}
after the first document and table: 9 entities, 8 relationships


### Stop and look

```cypher
MATCH (n) RETURN n
```

The organizations are there but named by their key — `ORG-NW`, `ORG-KG` — beside whatever
the board review called them in prose. A question naming "Northwind Energy" has nothing to
match on the table side yet.

You may also see a warning that some `WORKS_AT` relationships were **pruned**. That is the
ontology working: the extractor read the PDF and proposed a `WORKS_AT` between two
*organizations*, the declared pattern is `Person -> Organization`, so it was dropped. The
suggestion in that message about inverting the direction does not apply here.

In [5]:
for name, is_stub in await rag.query(
    "MATCH (o:Organization) RETURN o.name, o.is_stub ORDER BY o.name"
):
    print(f"  {name:22} {'placeholder' if is_stub else 'named by a source'}")

  Kestrel Grid           named by a source
  NORTHWIND ENERGY       named by a source
  ORG-KG                 placeholder
  ORG-NW                 placeholder


## 4. Another document, and the tables that name them

Two things happen to the entity count here, in opposite directions.

The **placeholders are filled in rather than duplicated** — `ORG-NW` becomes Northwind
Energy on the node that already existed. A reference node is written on create only, so a
row that merely points at an organization can never overwrite what that organization's own
source said. That is what makes the load order irrelevant, and on its own it would push the
count down.

At the same time **prose adds entities of its own**, so the number goes up by more than the
merge removes. The next cell shows the most interesting case.

`contracts.csv` is also the other shape worth seeing: **two links out of one row**, so one
contract points at a buyer and a seller.

In [6]:
await rag.ingest(str(WORK / "market_note.pdf"))
await rag.ingest(str(WORK / "organizations.csv"))
await rag.ingest(str(WORK / "contracts.csv"))
await report("finalize")
await show("after all five sources")


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]


Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 1069.16it/s]

Pruned 1 'BUYER' relationships due to (source, target) mismatch. Declared patterns: [('Contract', 'Organization')]. Observed (sample): [('Organization', 'Organization')]. If extraction looks correct, the pattern direction may be inverted.


Pruned 1 'SELLER' relationships due to (source, target) mismatch. Declared patterns: [('Contract', 'Organization')]. Observed (sample): [('Organization', 'Organization')]. If extraction looks correct, the pattern direction may be inverted.


Non-transient FalkorDB query failure: ResponseError: Attribute 'embedding' is already indexed


Non-transient FalkorDB query failure: ResponseError: Attribute 'embedding' is already indexed


Non-transient FalkorDB query failure: ResponseError: Attribute 'embedding' is already indexed


Non-transient FalkorDB query failure: ResponseError: Attribute 'text' is already indexed


Non-transient FalkorDB query failure: ResponseError: Attribute 'name' is already indexed


finalize: merged 0, embedded 3 entities
after all five sources: 10 entities, 10 relationships


In [7]:
print("contracts, both sides from one row:")
for buyer, contract, value, seller in await rag.query(
    """MATCH (b:Organization)<-[x:RELATES]-(k:Contract)-[y:RELATES]->(s:Organization)
       WHERE x.rel_type = 'BUYER' AND y.rel_type = 'SELLER'
         AND k.contracts__value_musd IS NOT NULL
       RETURN b.name, k.name, k.contracts__value_musd, s.name ORDER BY k.name"""
):
    print(f"  {buyer:18} buys {contract!r} (${value}M) from {seller}")

contracts, both sides from one row:
  Northwind Energy   buys 'Nordic transmission agreement' ($412.0M) from Kestrel Grid
  Northwind Energy   buys 'Peaking capacity option' ($58.5M) from Meridian Fuels


### The cost of declaring a label

Declaring `Contract` put it in front of the **extractor** as well as the loader. So when the
market note mentions a transmission agreement in prose, the extractor is now entitled to make
a `Contract` for it — and in our runs it does: a third node the CSV never keyed, with no
`value`, that is really the same agreement as one `contracts.csv` already wrote. Whether it
also gets edges depends on what the model extracts that run.

This is worth seeing rather than hiding, because the near-miss report **does not catch it**:
`canonical_key` does not join a name to a longer one containing it, and `why_same` returns
`None` for that shape. The resolver `finalize()` runs is the one thing that can: it is
shown both `Contract` nodes, asked whether they are one agreement, and if it says so the
prose node folds into the row `contracts.csv` keyed — check `resolved_duplicates` above.
A declared label is a capability you hand the extractor, and this is its bill.

In [8]:
rows = await rag.query(
    """MATCH (k:Contract)
       OPTIONAL MATCH (k)-[r:RELATES]->()
       RETURN k.name, k.contracts__value_musd, count(r) ORDER BY k.name"""
)
for name, value, edges in rows:
    # The CSV always writes a value; a node with none was minted by prose.
    origin = "from contracts.csv" if value is not None else "minted by prose"
    print(f"  {name:32} value={value!s:>6}  edges={edges}  {origin}")
if all(v is not None for _, v, _ in rows):
    print("  (the extractor did not mint an extra Contract this run)")

  Nordic transmission agreement    value= 412.0  edges=2  from contracts.csv
  Peaking capacity option          value=  58.5  edges=2  from contracts.csv
  transmission agreement           value=  None  edges=2  minted by prose


### One node, both halves

Each person carries columns from the CSV and, where a document named them, a description
from prose. `employees__age` is signed with the table that declared it, so no other source
can overwrite it — and a value the extractor reads from prose is unsigned by construction, so
the two cannot collide at all. Where two tables disagree, both values stay and `finalize()`
lists the property under `property_conflicts` rather than picking a winner.

The last column is the join itself: whether a **PDF chunk** mentions this person.

In [9]:
for name, age, title, in_pdf in await rag.query(
    """MATCH (p:Person)
       OPTIONAL MATCH (p)-[:MENTIONED_IN]->(c:Chunk)<-[:PART_OF]-(d:Document)
                      WHERE d.id ENDS WITH '.pdf'
       RETURN p.name, p.employees__age, p.employees__title, count(d) > 0
       ORDER BY p.name"""
):
    print(f"  {name:16} {age}  {title:28} {'named in a PDF' if in_pdf else 'table only'}")

  Johan Berg       29  Grid Analyst                 table only
  Maya Ellison     34  Engineer                     named in a PDF
  Priya Raman      39  Head of Regulatory Affairs   named in a PDF
  Tomas Reyes      47  Chief Financial Officer      named in a PDF


### A name the sources do not share

Everything above joined on an **exact** name: the CSV and the PDF both wrote "Priya Raman".
Real sources rarely oblige. An analyst call says "Ms. Raman", and no spelling rule can turn
that into a first name — so it becomes a second person, and the age is on one node while
what she said is on the other. Merging spellings cannot close that pair.

The resolver `finalize()` runs is what closes it. It is the same strategy that decides
*within* a document whether two mentions are one entity, shown the whole graph; and because
names differ more across sources than within one document — this pair embeds at 0.70, below
the within-document threshold of 0.80 — `finalize()`'s default asks the model from 0.6. It
sees the row's signed values (`employees: age 39, title Head of Regulatory Affairs`) beside
the call's sentence and their shared neighbour Kestrel Grid, and answers. On **YES** the
mention folds into the **row's** node — the table's id survives, its values are untouched,
and the prose description and the new mention move onto it. On **NO** both stay, and the
answer is remembered on the graph (`rejected_duplicates`), so the next `finalize()` does not
ask again — which is why the contract pair from the previous section is asked here once.


In [ ]:
ANALYST_CALL = """ANALYST CALL - KESTREL GRID, THIRD QUARTER
Speaking for Kestrel Grid's regulatory affairs team, Ms. Raman said the regulator's review of
the Meridian Fuels acquisition had closed without conditions, and that Kestrel Grid expects to
file the Danish tariff schedule in November. Asked about Northwind Energy, she said the
transmission agreement is unchanged."""

await rag.ingest(text=ANALYST_CALL, document_id="analyst_call.txt")
print("before finalize:", [n for (n,) in await rag.query("MATCH (p:Person) RETURN p.name ORDER BY p.name")])
await report("finalize")
for name, age, sources in await rag.query(
    """MATCH (p:Person)-[:MENTIONED_IN]->(:Chunk)<-[:PART_OF]-(d:Document)
       RETURN p.name, p.employees__age, collect(DISTINCT d.id) ORDER BY p.name"""
):
    print(f"  {name:16} {age}  {sorted(Path(s).name for s in sources)}")


## 5. The question neither half can answer alone

Which company missed its guidance is only in `board_review.pdf`. Who works there, and how
old they are, is only in `employees.csv`. Note the question never names Northwind.

In [10]:
await ask("Who works at the company that reported a revenue shortfall, and how old are they?")

Q: Who works at the company that reported a revenue shortfall, and how old are they?
A: Tomas Reyes, age 47, and Maya Ellison, age 34, work at Northwind Energy, the company that reported a revenue shortfall.


In [11]:
await ask("What is the average age of the employees?")

Q: What is the average age of the employees?
A: The average age of the employees is 37.25.


## 6. A corrected export arrives

A table is a **snapshot**, not an addition, so re-ingesting one already in the graph means
"this is the current state of it" — that is the update, and there is no separate call. Rows
that disappeared take their chunks with them, and any entity nothing else mentions goes too.

`employees_v2.csv` is the same export a quarter later: Maya is promoted, Johan has left, Lene
is new. It arrives under its own name, so `document_id="employees.csv"` says which table it
is: the mapping is found by that id, and the table — addressed by its name, not the path it
was read from — is re-synced with its path moving along.

In [12]:
print(open(WORK / "employees_v2.csv").read())

result = await rag.ingest(str(WORK / "employees_v2.csv"), document_id="employees.csv")
print(f"re-synced {result.records} rows: {result.chunks_deleted} chunks and "
      f"{result.entities_deleted} entities removed")
await report("finalize")
await show("after the corrected export")

employee_id,full_name,age,job_title,start_date,org_id
E-1,Maya Ellison,34,Principal Engineer,2019-04-01,ORG-NW
E-2,Tomas Reyes,47,Chief Financial Officer,2015-11-01,ORG-NW
E-3,Priya Raman,39,Head of Regulatory Affairs,2020-02-17,ORG-KG
E-5,Lene Dahl,31,Grid Analyst,2026-01-12,ORG-KG

re-synced 4 rows: 4 chunks and 1 entities removed


Non-transient FalkorDB query failure: ResponseError: Attribute 'embedding' is already indexed


Non-transient FalkorDB query failure: ResponseError: Attribute 'embedding' is already indexed


Non-transient FalkorDB query failure: ResponseError: Attribute 'embedding' is already indexed


Non-transient FalkorDB query failure: ResponseError: Attribute 'text' is already indexed


Non-transient FalkorDB query failure: ResponseError: Attribute 'name' is already indexed


finalize: merged 0, embedded 1 entities
after the corrected export: 10 entities, 10 relationships


In [13]:
await ask("Who works at the company that reported a revenue shortfall, and how old are they?")

Q: Who works at the company that reported a revenue shortfall, and how old are they?
A: Tomas Reyes, age 47, and Maya Ellison, age 34, work at Northwind Energy, the company that reported a revenue shortfall.


In [14]:
await ask("What is the average age of the employees?")

Q: What is the average age of the employees?
A: The average age of the employees is 37.75.


Maya's title changed, Johan is gone, Lene is new, and the average age moved with them. The
PDFs were never re-read, and the organizations' own columns were never at risk, because
every structured property belongs to exactly one source.

**Next:** `tickets.csv` in the fixtures is a table of free text — a case where you *want* the
prose path. Pass a loader to say so: `await rag.ingest(path, loader=TextLoader())`.

In [15]:
await rag.close()
shutil.rmtree(WORK.parent, ignore_errors=True)